# 📓 04 — 可视化与营销策略输出## 目标- 制作 3D 散点图、雷达图、营销漏斗- 对每个用户群体输出**可落地的营销策略**## 输出所有图保存到 `images/`，策略表可复制到 README 与简历。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("../src")
from visualization import plot_rfm_radar
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)


## Step 1: 加载聚类结果

In [ ]:
rfm = pd.read_pickle("../data/processed/rfm_clustered.pkl")
print(f"客户数: {len(rfm)}")
rfm.head()


## Step 2: 各群体规模与营收贡献

In [ ]:
segment_summary = (
    rfm.groupby("Segment_Name")
    .agg(
        Count=("Cluster", "size"),
        Avg_Recency=("Recency", "mean"),
        Avg_Frequency=("Frequency", "mean"),
        Avg_Monetary=("Monetary", "mean"),
        Total_Revenue=("Monetary", "sum"),
    )
    .round(2)
)
segment_summary["Pct_Customers"] = (
    segment_summary["Count"] / segment_summary["Count"].sum() * 100
).round(1)
segment_summary["Pct_Revenue"] = (
    segment_summary["Total_Revenue"] / segment_summary["Total_Revenue"].sum() * 100
).round(1)
segment_summary = segment_summary.sort_values("Pct_Revenue", ascending=False)
print(segment_summary)

# 帕累托图
fig, ax1 = plt.subplots(figsize=(10, 5))
sns.barplot(x=segment_summary.index, y=segment_summary["Pct_Revenue"],
            ax=ax1, palette="viridis", order=segment_summary.index)
ax1.set_ylabel("Revenue Share (%)")
ax1.set_title("Revenue Contribution by Segment (Pareto View)")
for i, v in enumerate(segment_summary["Pct_Revenue"]):
    ax1.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("../images/revenue_pareto.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 3: 3D 散点图（交互式）

In [ ]:
fig = px.scatter_3d(
    rfm.reset_index(),
    x="Recency", y="Frequency", z="Monetary",
    color="Segment_Name",
    title="RFM 3D Customer Segmentation",
    labels={"Recency": "Recency (days)", "Frequency": "Frequency", "Monetary": "Monetary (GBP)"},
    opacity=0.6, size_max=8,
)
fig.update_layout(width=900, height=650)
fig.write_html("../images/rfm_3d.html")
fig.show()


## Step 4: 各群体 RFM 雷达图

In [ ]:
# 把 R/F/M 转成 1-5 的归一化得分，便于统一刻度
rfm_norm = rfm.copy()
rfm_norm["R_norm"] = 1 - (rfm_norm["Recency"] / rfm_norm["Recency"].max()) * 4 + 1
rfm_norm["F_norm"] = (rfm_norm["Frequency"] / rfm_norm["Frequency"].max()) * 4 + 1
rfm_norm["M_norm"] = (rfm_norm["Monetary"] / rfm_norm["Monetary"].max()) * 4 + 1

radar_data = (
    rfm_norm.groupby("Segment_Name")[["R_norm", "F_norm", "M_norm"]]
    .mean()
    .round(2)
)
radar_data.columns = ["R", "F", "M"]
print(radar_data)

plot_rfm_radar(radar_data)
plt.savefig("../images/rfm_radar.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 5: 营销策略矩阵

In [ ]:
strategy = {
    "Champions": {
        "特征": "高 M + 低 R + 高 F",
        "占比": "8-12%",
        "策略": [
            "VIP 专属客户经理 1v1 服务",
            "新品/限量款优先购",
            "邀请参与品牌共创、用户调研",
            "推荐裂变激励（拉新奖励）",
        ],
        "预期": "留存率 +20%，客单价 +15%",
    },
    "At-Risk VIP": {
        "特征": "高 M + 高 R（曾经高频高额，最近没来）",
        "占比": "5-8%",
        "策略": [
            "大额召回券（≥ 消费金额的 20%）",
            "专属客服电话回访",
            "调研流失原因（问卷/访谈）",
            "限时 7 日 VIP 体验续期",
        ],
        "预期": "召回率 15-25%",
    },
    "New Customers": {
        "特征": "低 R + 低 F + 低 M（刚来）",
        "占比": "18-25%",
        "策略": [
            "首单 7 日内推送 8 折复购券",
            "自动化欢迎邮件 + 品类推荐",
            "加入新人社群、引导签到",
            "提供满减门槛（提升客单价）",
        ],
        "预期": "30 日复购率 +30%",
    },
    "Hibernating": {
        "特征": "高 R + 中 F + 中 M（沉默用户）",
        "占比": "12-18%",
        "策略": [
            "唤醒邮件/短信 + 大额折扣",
            "推送热门爆款 + 个性化推荐",
            "限时免费配送刺激下单",
            "若仍不响应，移入低优先级池",
        ],
        "预期": "唤醒率 5-10%",
    },
    "Regular": {
        "特征": "中等 R / F / M",
        "占比": "30-40%",
        "策略": [
            "标准自动化营销流（节日/活动）",
            "推送满减券提升客单价",
            "个性化商品推荐",
            "鼓励注册会员、积分体系",
        ],
        "预期": "ARPU +8-12%",
    },
}

import pprint
pprint.pprint(strategy)


## Step 6: 营销漏斗

In [ ]:
# 模拟一个从认知 → 首单 → 复购 → 高价值的转化漏斗
funnel = pd.DataFrame({
    "Stage": ["总客户", "活跃客户", "首单客户", "复购客户", "高价值客户"],
    "Count": [
        len(rfm),
        (rfm["Recency"] < 90).sum(),
        (rfm["Frequency"] >= 1).sum(),
        (rfm["Frequency"] >= 2).sum(),
        (rfm["Monetary"] > rfm["Monetary"].quantile(0.75)).sum(),
    ],
})

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(funnel["Stage"][::-1], funnel["Count"][::-1], color="steelblue")
for bar, v in zip(bars, funnel["Count"][::-1]):
    ax.text(v + 500, bar.get_y() + bar.get_height()/2, f"{v:,}", va="center")
ax.set_title("Customer Conversion Funnel")
ax.set_xlabel("Customers")
plt.tight_layout()
plt.savefig("../images/conversion_funnel.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 7: 导出营销策略表

In [ ]:
import os
rows = []
for seg, info in strategy.items():
    rows.append({
        "Segment": seg,
        "Characteristic": info["特征"],
        "Strategy": " | ".join(info["策略"]),
        "Expected_Impact": info["预期"],
    })
strategy_df = pd.DataFrame(rows)
strategy_df.to_csv("../data/processed/marketing_strategy.csv", index=False, encoding="utf-8-sig")
print("已导出 marketing_strategy.csv")
strategy_df


## ✅ 本节小结- 完成 5 个用户群体的**画像、占比、营收贡献**分析- 输出 3D 散点图、雷达图、漏斗图、帕累托图（保存在 `images/`）- 为每个群体制定了**4 条具体可落地的营销策略**- 策略表已导出为 `marketing_strategy.csv`，可直接复用## 📌 接下来1. 把所有图截图放进 `README.md`2. 在 GitHub 创建 repo 并推送3. 写一段 LLM 友好的项目描述（给 HR 一眼看懂）